In [60]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score

pd.set_option('display.max_columns', None)
prefix = "../data/processed/"

In [61]:
#Regular games 
df = (pd.read_csv(prefix + "TeamStatisticsFrom2010RegRoll10.csv")).dropna()

In [62]:
#Roll features
fea = []
for col in df.columns:
    if 'roll' in col: 
        fea.append(col)
        print(col)

teamScore_roll10
fieldGoalsPercentage_roll10
threePointersPercentage_roll10
freeThrowsPercentage_roll10
reboundsOffensive_roll10
reboundsDefensive_roll10
assists_roll10
turnovers_roll10
steals_roll10
blocks_roll10
plusMinusPoints_roll10
win_roll10


In [63]:
#Merging home and away teams by gameId 
df_m = df.merge(
    df,
    on='gameId',
    suffixes=('_home', '_away')
)

# Keep only the home team's perspective (home == 1) to avoid mirrored duplicates
df_m = df_m[df_m['home_home'] == 1]
df_m = df_m[df_m['teamId_home'] != df_m['teamId_away']]

#Generating a list of columns from the differences in the rolling averages from the home and away teams
diff_cols = []
for col in fea: 
    diff_name = f'{col}_diff'
    df_m[diff_name] = df_m[f'{col}_home'] - df_m[f'{col}_away']
    diff_cols.append(diff_name)


In [64]:
#Initial Test and Train split
xg_tr, xg_te, yg_tr, yg_te = train_test_split(df_m, df_m['win_home'], random_state=17, test_size=0.2)

In [68]:
#PyGAM initialization 
import scipy.sparse
from sklearn.preprocessing import PowerTransformer
from sklearn.naive_bayes import GaussianNB

def to_array(self):
    return self.toarray()

scipy.sparse.spmatrix.A = property(to_array)

from pygam import LogisticGAM, s, l, te

#Features for the pyGam
py_fea = [
    'fieldGoalsPercentage_roll10_diff','plusMinusPoints_roll10_diff', 'plusMinusPoints_roll10_away', 
    'fieldGoalsPercentage_roll10_diff', 'turnovers_roll10_home'
    ]


#Pipeline for pygam
pipe_g = Pipeline(steps=[
    ('scale', StandardScaler()),
    ('gam', LogisticGAM(
        s(0, 100) + l(1) + l(2) + s(3) + l(4) + te(0,1), max_iter=300, 
        ))
])


In [69]:
#KFold Validation step
#Loop that iterates over each column in the above list, fits, and calculates the accuracy
#Only models/variables that yield results better than chance are reported

kf       = KFold(n_splits=5)
stat_ary = np.zeros((3, 5))
    
for i, (train_ix, test_ix) in enumerate(kf.split(xg_tr)):
   xr = xg_tr[py_fea].iloc[train_ix].values
   yr = yg_tr.iloc[train_ix]
   xh = xg_tr[py_fea].iloc[test_ix].values
   yh = yg_tr.iloc[test_ix]
   pipe_g.fit(xr,yr)

   y_pred = pipe_g['gam'].predict(xh)
   acc    = 1.0*sum(y_pred==yh) / len(y_pred)
   prob   = pipe_g['gam'].predict_proba(xh)
   roc    = roc_auc_score(yh, prob)
   prec   = precision_score(yh, y_pred)

   stat_ary[0][i] = acc
   stat_ary[1][i] = roc
   stat_ary[2][i] = prec

print('Acc Mean(STD) %.3f(%.3f)\nROC Mean(STD) %.3f(%.3f)\nPrec Mean(STD) %.3f(%.3f)'%(stat_ary[0].mean(), stat_ary[0].std(), stat_ary[1].mean(), stat_ary[1].std(),stat_ary[2].mean(), stat_ary[2].std()))



/home/mars/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pygam/pygam.py:631: RuntimeWarning: overflow encountered in square
  self.link.gradient(mu, self.distribution) ** 2
/home/mars/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pygam/links.py:137: RuntimeWarning: overflow encountered in divide
  return dist.levels / (mu * (dist.levels - mu))
/home/mars/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pygam/links.py:137: RuntimeWarning: divide by zero encountered in divide
  return dist.levels / (mu * (dist.levels - mu))
/home/mars/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pygam/pygam.py:631: RuntimeWarning: invalid value encountered in multiply
  self.link.gradient(mu, self.distribution) ** 2
/home/mars/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pygam/links.py:121: RuntimeWarning: overflow encountered in exp
  elp = np.exp(lp)
/home/mars/miniconda3/envs/erdos_ds_environment/lib/pyt

Acc Mean(STD) 0.630(0.009)
ROC Mean(STD) 0.676(0.009)
Prec Mean(STD) 0.701(0.017)


In [59]:
#Full model test
pipe_g.fit(xg_tr,yg_tr)

y_pred = pipe_g['gam'].predict(xg_te)
acc    = 1.0*sum(y_pred==yg_te) / len(y_pred)
prob   = pipe_g['gam'].predict_proba(xg_te)
roc    = roc_auc_score(yg_te, prob)
prec   = precision_score(yg_te, y_pred)

ValueError: could not convert string to float: '2021-12-21'